In [215]:
from __future__ import annotations
import hashlib
import networkx as nx

import netbiol3 as nb
a = nb.Abasy(db='abasy_internal', rest=False)

In [222]:
class GraphObserver:

    """A class to observe changes in a graph."""

    def __init__(self, G: nx.Graph | nb.RegNet, data: bool = False) -> None:
        """
        Initialize the GraphObserver class.
        
        Args:
            G (nx.Graph | nb.RegNet): The graph to observe.
            data (bool): If True, the node/edge data will be considered when computing the hash.
        
        """
        self.G = G
        self.data = data
        self.graph_hash = self._compute_hash(self.G)

    def _compute_hash(self, G: nx.Graph | nb.RegNet) -> str:
        """
        Compute the hash of the graph.

        Args:
            G (nx.Graph | nb.RegNet): The graph to compute the hash.
            data (bool): If True, the node/edge data will be considered when computing the hash.

        Returns:
            str: The hash of the graph.
        """

        hash = hashlib.sha1(
            f"nodes: {G.nodes(data=self.data)},\
            edges: {G.edges(data=self.data)}".encode('utf-8')
            )
        return hash.hexdigest()

    def changed(self, G: nx.Graph | nb.RegNet = None, update_G: bool = False) -> bool:
        """
        Check if G has changed with reference to the last call.

        Args:
            G (nx.Graph | nb.RegNet): The graph to check. If None, the original graph will be used.
            update_G (bool): If True, the graph will be updated to G. If False, the graph will not be updated.

        Returns:
            bool: True if the graph has changed, False otherwise.

        Raises:
            ValueError: If G is None and update_G is True.

        Note:
            If G is None and update_G is False, the original graph will be used (default behavior).

            The following table shows the behavior of the function:
            hash_org_G: hash of the original graph
            hash_new_G: hash of the new graph

            | G | update_G | return | update graph | update hash |
            |---|----------|--------|--------------|-------------|
            | None | False | hash_org_G == new_hash_org_G | False | True |
            | not None | True | hash_org_G == hash_new_G | True | True |
            | not None | False | hash_org_G == hash_new_G | False | False |
            | None | True | ValueError | False | False |
        """
        if update_G and G is None:
            raise ValueError("G cannot be None if update_G is True")

        tmp_G = self.G if G is None else G
        new_hash = self._compute_hash(tmp_G)

        if new_hash != self.graph_hash:
            change_flag = True

            if G is None:
                # update the hash of the original graph
                self.graph_hash = new_hash

            elif update_G:
                # G and update_G
                self.G = G
                self.graph_hash = new_hash
            
            # if not update_G and G is not None, no need to update the hash

        else:
            change_flag = False

            if update_G:
                # G and update_G
                self.G = G
                # no need to update the hash

        return change_flag
    

In [225]:
import pytest

def test_graph_observer():
    """Test the GraphObserver class."""

    # Test the GraphObserver class with a simple graph.
    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G)
    assert graph_observer.changed() is False
    assert graph_observer.changed() is False
    
    G.remove_node('crp')
    assert graph_observer.changed() is True
    assert graph_observer.changed() is False

    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G) # data=False by default
    G.nodes['crp']['NDA'] = 'Not DNA class anymore!!!'
    assert graph_observer.changed() is False
    assert graph_observer.changed() is False

    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G, data=True)
    G.nodes['crp']['NDA'] = 'Not DNA class anymore!!!'
    assert graph_observer.changed() is True
    assert graph_observer.changed() is False

    # Test the GraphObserver class when passing a new graph.
    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G)
    G = G.copy() # not the same object
    G.remove_node('crp')
    assert graph_observer.changed() is False
    assert graph_observer.changed() is False

    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G)
    G = G.copy() # not the same object
    G.remove_node('crp')
    assert graph_observer.changed(G) # update_G=False
    assert graph_observer.changed() is False # still looks at the original graph
    assert 'crp' in graph_observer.G.nodes() # still looks at the original graph

    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G)
    G = G.copy() # not the same object
    G.remove_node('crp')
    assert graph_observer.changed(G, update_G=True)
    assert graph_observer.changed() is False
    assert 'crp' not in graph_observer.G.nodes()

    G = a.regnet('511145_v2003_sRDB01')
    graph_observer = GraphObserver(G)
    G = G.copy() # not the same object
    G.remove_node('crp')    
    with pytest.raises(ValueError):
        graph_observer.changed(update_G=True) # inconsistent
    assert graph_observer.changed() is False
    assert 'crp' in graph_observer.G.nodes()

In [226]:
test_graph_observer()

In [ ]:
import pytest

import netbiol3 as nb
a = nb.Abasy()

class TestGraphObserver:
    @pytest.fixture
    def graph_observer(self):
        G = a.regnet('511145_v2003_sRDB01')
        return GraphObserver(G)

    def test_changed_false(self, graph_observer):
        assert not graph_observer.changed()
        assert not graph_observer.changed()

    def test_changed_true(self, graph_observer):
        graph_observer.G.remove_node('crp')
        assert graph_observer.changed()
        assert not graph_observer.changed()

    def test_changed_with_data_false(self):
        G = a.regnet('511145_v2003_sRDB01')
        graph_observer = GraphObserver(G, data=False) # data=False by default
        G.nodes['crp']['NDA'] = 'Not DNA class anymore!!!'
        assert not graph_observer.changed()
        assert not graph_observer.changed()

    def test_changed_with_data_true(self):
        G = a.regnet('511145_v2003_sRDB01')
        graph_observer = GraphObserver(G, data=True)
        G.nodes['crp']['NDA'] = 'Not DNA class anymore!!!'
        assert graph_observer.changed()
        assert not graph_observer.changed()

    def test_changed_with_new_graph_false(self, graph_observer):
        G = graph_observer.G.copy() # not the same object
        G.remove_node('crp')
        assert not graph_observer.changed()
        assert not graph_observer.changed()

    def test_changed_with_new_graph_true_update_G_false(self, graph_observer):
        G = graph_observer.G.copy()
        G.remove_node('crp')
        assert graph_observer.changed(G) # just compare the hashes
        assert not graph_observer.changed() # still looks at the original graph
        assert 'crp' in graph_observer.G.nodes() # still looks at the original graph

    def test_changed_with_new_graph_true_update_G_true(self, graph_observer):
        G = graph_observer.G.copy() # not the same object
        G.remove_node('crp')
        assert graph_observer.changed(G, update_G=True)
        assert not graph_observer.changed()
        assert 'crp' not in graph_observer.G.nodes()

    def test_changed_with_inconsistent_update_G(self, graph_observer):
        G = graph_observer.G.copy() # not the same object
        G.remove_node('crp')
        with pytest.raises(ValueError):
            graph_observer.changed(update_G=True)
        assert not graph_observer.changed()
        assert 'crp' in graph_observer.G.nodes()


## norm

In [157]:
import hashlib
import pandas as pd

# ths is the good :D

class NormObserver:
    """A class to observe changes in the normalization strategy."""
    def __init__(self, norm):
        """
        Initialize the NormObserver class.
        
        Args:
            norm (None | str | pd.Series): The norm to observe.

        """
        self.norm = norm
        self.norm_hash = self._compute_hash()

    def _hash(self, str_norm):
        """
        Compute the SHA-1 hash of a string.
        Implemented to DRY the code.

        Args:
            str_norm (str): The string to compute the hash.

        Returns:
            str: The hash of the string.
        """
        hash_object = hashlib.sha1(str_norm.encode('utf-8'))
        return hash_object.hexdigest()

    def _compute_hash(self):
        """
        Compute the SHA-1 hash of the current norm value
        
        Args:
            norm (None | str | pd.Series): The norm to compute the hash.
        
        Returns:
            str: The hash of the norm. If norm is None, return None.
        """
        if self.norm is None:
            return None
        
        elif isinstance(self.norm, str):
            str_norm = self.norm
            return self._hash(str_norm)

        elif isinstance(self.norm, pd.Series):
            # convert it to a flat string to be hashed
            # self.norm.to_string(index=True, dtype=True, name=True, length=True, header=True)
            str_norm = f"pd.Series: {self.norm}"
            return self._hash(str_norm)

    def change(self):
        """
        Check if norm has changed with reference to the last call.

        Returns:
            bool: True if the norm has changed, False otherwise.
        """
        new_hash = self._compute_hash()
        if new_hash != self.norm_hash:
            change_flag = True
            self.norm_hash = new_hash
        else:
            change_flag = False
        return change_flag


In [158]:
import pandas as pd


def test_norm_observer():
    # Test with string norm
    norm = "biol"
    norm_observer = NormObserver(norm)
    assert norm_observer.change() is False
    assert norm_observer.change() is False
    norm_observer.norm = "netheory"
    assert norm_observer.change() is True
    assert norm_observer.change() is False

    # Test with None norm
    norm_observer = NormObserver(None)
    assert norm_observer.change() is False
    assert norm_observer.change() is False
    norm_observer.norm = "biol"
    assert norm_observer.change() is True
    assert norm_observer.change() is False

    # Test with pd.Series norm
    data = {'a': 0.8, 'b': 0.01, 'c': 1}
    series_norm = pd.Series(data)
    norm_observer = NormObserver(series_norm)
    assert norm_observer.change() is False
    assert norm_observer.change() is False
    update_ser = {'a': 10, 'b': 11, 'c': 12}
    series_norm = pd.Series(update_ser)
    norm_observer.norm = series_norm
    assert norm_observer.change() is True
    assert norm_observer.change() is False

    # Test with other objects
    norm_observer = NormObserver(42)
    assert norm_observer.change() is False
    assert norm_observer.change() is False
    norm_observer.norm = 43
    # ignores the change because the norm is not None, a string or a pd.Series
    assert norm_observer.change() is False
    assert norm_observer.change() is False


In [159]:
test_norm_observer()

In [219]:
import pandas as pd
import pytest

class TestNormObserver:
    @pytest.fixture
    def norm_observer(self):
        return NormObserver(None)
    
    @pytest.fixture
    def series_norm():
        data = {'a': 0.8, 'b': 0.01, 'c': 1}
        return pd.Series(data)
    
    def test_change_with_string_norm(self, norm_observer):
        norm_observer.norm = "biol"
        assert norm_observer.change() is True
        assert norm_observer.change() is False
        
        norm_observer.norm = "netheory"
        assert norm_observer.change() is True
        assert norm_observer.change() is False
        
    def test_change_with_none_norm(self, norm_observer):
        norm_observer.norm = None
        assert norm_observer.change() is True
        assert norm_observer.change() is False
        
        norm_observer.norm = "biol"
        assert norm_observer.change() is True
        assert norm_observer.change() is False
        
    def test_change_with_pd_series_norm(self, series_norm):
        norm_observer.norm = series_norm
        assert norm_observer.change() is True
        assert norm_observer.change() is False

        norm_observer.norm = None
        assert norm_observer.change() is True
        assert norm_observer.change() is False

        norm_observer = NormObserver(series_norm)
        assert norm_observer.change() is False
        update_ser = {'a': 10, 'b': 11, 'c': 12}
        norm_observer.norm = pd.Series(update_ser)
        assert norm_observer.change() is True
        assert norm_observer.change() is False
        
    def test_change_with_other_objects(norm_observer):
        norm_observer.norm = 42 # ignores the change because the norm is not None, a string or a pd.Series
        # TODO: check if this is the desired behavior, validating the input only in the norm_setter (Srucutre)
        assert norm_observer.change() is False
        assert norm_observer.change() is False


Overwriting test_list.py


class GraphObserver:

    """A class to observe changes in a graph."""

    def __init__(self, G: nx.Graph | nb.RegNet, data: bool = False):
        """
        Initialize the GraphObserver class.
        
        Args:
            G (nx.Graph | nb.RegNet): The graph to observe.
            data (bool): If True, the node/edge data will be considered when computing the hash.
        
        Notes:
            The computation of the hash is considerately slower when data is True.
        """
        self.data = data
        
        self.graph_hash = self._compute_hash(G)

    def _compute_hash(self, G):
        """
        Compute the hash of the graph.

        Args:
            G (nx.Graph | nb.RegNet): The graph to compute the hash.

        Returns:
            str: The hash of the graph.
        """

        if self.data:
            nodes = [(n, OrderedDict(sorted(d.items()))) for n, d in G.nodes(data=True)]
            edges = [(u, v, OrderedDict(sorted(d.items()))) for u, v, d in G.edges(data=True)]
        else:
            nodes = list(G.nodes()) # can't pickle dict_keys objects in pickle.dumps(G) otherwise
            edges = list(G.edges())

        # Create a new graph
        H = nx.DiGraph()
        H.add_nodes_from(nodes)
        H.add_edges_from(edges)

        # Serialize the new graph
        serialized_graph = pickle.dumps(H)
        hash_object = hashlib.sha256(serialized_graph)
        return hash_object.hexdigest()

    def changed(self, G):
        """
        Compares the hash of G with the hash of the graph when the GraphObserver was initialized.

        Returns:
            bool: True if the graph has changed, False otherwise.
        """
        new_hash = self._compute_hash(G)
        return new_hash != self.graph_hash
    

# Use pickle and OderedDict to compute the hash of a graph with data
from __future__ import annotations
import pickle
import hashlib
import networkx as nx
from collections import OrderedDict

import netbiol3 as nb
a = nb.Abasy()

class GraphObserver:

    """A class to observe changes in a graph."""

    def __init__(self, G: nx.Graph | nb.RegNet, data: bool = False):
        """
        Initialize the GraphObserver class.
        
        Args:
            G (nx.Graph | nb.RegNet): The graph to observe.
            data (bool): If True, the node/edge data will be considered when computing the hash.
        
        Notes:
            The computation of the hash is considerately slower when data is True.
        """
        self.G = G
        self.data = data
        self.graph_hash = self._compute_hash()

    def _compute_hash(self):
        """
        Compute the hash of the graph.

        Args:
            G (nx.Graph | nb.RegNet): The graph to compute the hash.

        Returns:
            str: The hash of the graph.
        """

        if self.data:
            nodes = [(n, OrderedDict(sorted(d.items()))) for n, d in self.G.nodes(data=True)]
            edges = [(u, v, OrderedDict(sorted(d.items()))) for u, v, d in self.G.edges(data=True)]
        else:
            nodes = list(self.G.nodes()) # can't pickle dict_keys objects in pickle.dumps(G) otherwise
            edges = list(self.G.edges())

        # Create a new graph
        H = nx.DiGraph()
        H.add_nodes_from(nodes)
        H.add_edges_from(edges)

        # Serialize the new graph
        serialized_graph = pickle.dumps(H)
        hash_object = hashlib.sha1(serialized_graph)
        return hash_object.hexdigest()

    def changed(self):
        """
        Compares the hash of G with the hash of the graph when the GraphObserver was initialized.

        Returns:
            bool: True if the graph has changed, False otherwise.
        """
        new_hash = self._compute_hash()
        return new_hash != self.graph_hash
    
